# Lung Nodule 3D — Colab runner

Runtime -> Change runtime type -> **T4 GPU**.

This notebook: (1) clones the repo, (2) installs deps, (3) gets data, (4) trains the 3D CNN, (5) runs the Haralick+ANN baseline, (6) shows metrics + Grad-CAM.

Free tier: use `--model.depth 10` and `patch_size 48`. Colab Pro (A100/L4, more RAM): `--model.depth 18`, larger batch.

In [ ]:
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
!nvidia-smi -L

In [ ]:
# --- 1. get the code ---
# Option A: clone your GitHub repo
# !git clone https://github.com/<you>/lung-nodule-3d.git
# %cd lung-nodule-3d
# Option B: mount Drive and copy the folder there
from google.colab import drive; drive.mount('/content/drive')
%cd /content/drive/MyDrive/lung-nodule-3d

In [ ]:
# --- 2. deps (torch/torchvision already on Colab) ---
!pip -q install pylidc SimpleITK scikit-image monai pyyaml

In [ ]:
# --- 3a. FAST PATH: sanity-check the whole pipeline on synthetic data (~3 min on T4) ---
!python scripts/make_synthetic_data.py --n 400 --patch-vox 64
!python -m src.engine.train --train.epochs 15 --output.run_name syn_demo

In [ ]:
# --- 3b. REAL DATA: pre-extracted LIDC nodule patches ---
# Put a patches.zip (patches/*.npy + manifest.csv) on your Drive, or download from
# a Kaggle dataset with the Kaggle API, then unzip into data/processed/.
# from google.colab import files; ...
# !unzip -q /content/drive/MyDrive/lidc_patches.zip -d data/processed/
#
# OR extract them yourself from raw LIDC DICOM on Drive (slow, ~1-2h for full set):
# configure ~/pylidc.conf to point at the DICOM dir, then:
# !python -m src.data.extract_patches --out data/processed --patch-mm 40 --patch-vox 64 --limit 200

In [ ]:
# --- 4. train the 3D CNN on real patches ---
!python -m src.engine.train \
  --train.epochs 60 --train.batch_size 32 \
  --model.depth 18 --data.patch_size 48 \
  --output.run_name resnet3d_lidc

In [ ]:
# --- 5. baseline: original 2023 Haralick + ANN method, same split ---
!python -m src.baseline.haralick_ann

In [ ]:
# --- 6. evaluate + figures + Grad-CAM ---
!python -m src.engine.evaluate --run artifacts/resnet3d_lidc
import pandas as pd, json
print(json.load(open('artifacts/resnet3d_lidc/test_metrics.json')))
from IPython.display import Image; Image('artifacts/resnet3d_lidc/roc.png')